# Spike — how many YC companies can be found in SEC Form D filings?

**This notebook exists to produce one number**, before anything is built on it.

A US company raising a private round files **Form D** with the SEC: issuer, date,
offering amount — free, official, and the closest thing to real funding data this
project can reach. What nobody knows yet is *how many* YC companies can actually be
tied to a filing.

**The method is deliberately naive:** exact name matching, normalised only for
punctuation and legal suffixes. It will miss renamed companies, holding-company
filers and everyone who never filed. **That failure rate is the result.** If the
number is low, the branch closes honestly and the plan changes; if it is high, it
justifies building real entity resolution next.

The counting refuses to flatter itself: two filers with the same name are
*ambiguous*, never "the first one", and a failed request is an *error*, never
"no filings".

Nothing here costs money, and no key is needed.

In [ ]:
# Parameters (papermill overrides these).
sample_size = 200          # companies to look up
out_dir = "data/spikes"    # where the report lands
dataset = ""               # empty = newest dated dataset in data/
pause = 0.15               # seconds between requests; SEC asks for < 10/second

In [ ]:
import json
from pathlib import Path

import pandas as pd

from yc_scouter import config, sec_edgar

path = Path(dataset) if dataset else (
    config.latest_dated("ai", "parquet") or config.latest_dated("base", "parquet")
)
if path is None:
    raise SystemExit("No dataset found. Run File 1 (Base) first.")

df = pd.read_parquet(path)[["id", "name"]]
companies = sec_edgar.take_sample(df.to_dict("records"), sample_size)
print(f"Dataset: {path.name}  ·  companies: {len(df)}  ·  sample: {len(companies)}")
print(f"Identifying as: {sec_edgar.user_agent()}")

## The measurement

One request per company, paced under SEC's rate guidance. A sample of 200 takes a
couple of minutes.

In [ ]:
report = sec_edgar.measure_coverage(companies, pause=pause)
counts = report["counts"]
total = report["sample_size"] or 1

for kind in ("matched", "ambiguous", "none", "error"):
    print(f"{kind:>10}: {counts[kind]:>4}  ({counts[kind] / total:.1%})")

crowded = [r for r in report["rows"] if r["others"] >= 5]
print(f"\nNames colliding with 5+ other filers: {len(crowded)}"
      "  (a name-only match there deserves less trust)")

## What the number means

- **matched** — one filer carries this exact name. Not proof it is the same
  company; a name-only match is evidence, not identity.
- **ambiguous** — several filers share the name. Choosing one would invent a fact.
- **none** — no filer by that name. Could be: never raised, raised on a SAFE that
  was not filed, non-US, or renamed.
- **error** — the request failed. **Says nothing about the company.**

In [ ]:
out = Path(out_dir)
out.mkdir(parents=True, exist_ok=True)
target = out / f"formd_coverage_{report['generated_at']}.json"
target.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Wrote {target}")

pd.DataFrame(report["rows"]).head(20)